# 01 · Network metrics  (Paper Table 2, "NWMetrics")

Recomputes Table NWMetrics directly from the **real Java network generators**
(`src/network/*.java`), not a Python reimplementation. Uses
`src/dynamics/NetworkDump.java` — a standalone dumper that replays the exact
`randomGenerator.init(seed)` + network-constructor call that
`OpinionDynamics.main()` uses, so the dumped edge list is byte-identical to the
step-0 topology the actual simulation builds.

**Topology parameters match the current degree-aligned sweep**
(`<k> ~= 4`, i.e. the same `network_params` as the live
`results/summary/**/config.yaml` conditions) — CNNR p=0.5 r=0.01; HK m=2 A=1
pt=0.0/0.3; WS K=4 beta=0.1; ER p=0.002. The previous version of this notebook
used an ad-hoc Python re-implementation with stale, **pre**-degree-alignment
parameters (CNNR p=0.3, HK m=3, ER p=0.003) and a crude Hill estimator for the
power-law exponent; it did not reproduce Table 2 at all. This version was
verified by hand against Table 2 before being committed (see project memory
`nwmetrics-table-method`).

**Methodology** (agreed 2026-06-11):
- Average degree: undirected $2E/N$, mean over 20 seeds.
- Average path length / diameter: `networkx` on the undirected largest
  connected component, mean over 5 seeds (expensive at $N{=}1000$).
- Average clustering: `nx.average_clustering` on the **directed** graph
  (Fagiolo 2007), mean over 20 seeds.
- Power-law exponent: `powerlaw` package (Alstott 2014), discrete MLE,
  $x_{\min}$ auto-selected via KS, fit on in-degree, mean over 5 seeds.
  Reported only for CNNR/HK; WS/ER are reported as `---` in the table (see the
  honesty caveat at the end of this notebook — WS/ER fit a much steeper,
  non-heavy-tailed exponent, consistent with the paper's framing that they
  lack a heavy tail, so a "power-law exponent" for them would misleadingly
  imply a heavy tail exists).

Writes intermediate edge-list dumps to `results/.netdump/` (gitignored,
regenerated on each run — nothing precomputed is shipped).

In [1]:
# --- bootstrap: run from anywhere; cd to repo root (has results/ and src/) ---
import os
for _ in range(5):
    if os.path.isdir('results') and os.path.isdir('src'):
        break
    os.chdir('..')
print('working dir:', os.getcwd())

working dir: /home/tomoyatakeda/document/research/manipulation-backfire-model


## 1. Compile `NetworkDump` and dump the initial networks

Temporarily overwrites the repo-root `config.yaml` (one topology at a time),
invokes `java dynamics.NetworkDump <nSeeds> <outDir>`, then restores the
original `config.yaml` via `git checkout` — `config.yaml` is a tracked file
that the live simulation pipeline (`run.sh`/`run_all.sh`) also uses, so it must
be left exactly as found even if a cell errors partway through.

In [2]:
import glob
import subprocess
import yaml

N_SEEDS = 20
NETDUMP_DIR = os.path.abspath('results/.netdump')
CLASSPATH = ':'.join(glob.glob('lib/**/*.jar', recursive=True) + ['bin'])

# network_params match the live degree-aligned sweep's config.yaml files
# (results/summary/{CNNR,HolmeKim,WS,ER}/**/config.yaml), i.e. <k> ~= 4 for all five.
TOPOLOGY_CONFIGS = {
    'CNNR':        {'topology': 'CNNR',     'network_params': {'p': 0.5, 'r': 0.01}},
    'HK (pt=0.0)': {'topology': 'HolmeKim', 'network_params': {'m': 2, 'A': 1, 'pt': 0.0}},
    'HK (pt=0.3)': {'topology': 'HolmeKim', 'network_params': {'m': 2, 'A': 1, 'pt': 0.3}},
    'WS':          {'topology': 'WS',       'network_params': {'K': 4, 'beta': 0.1}},
    'ER':          {'topology': 'ER',       'network_params': {'p': 0.002}},
}

subprocess.run(['javac', '-cp', CLASSPATH, '-d', 'bin'] + glob.glob('src/**/*.java', recursive=True),
                check=True)

def _dir_name(topo_name):
    return topo_name.replace(' ', '_').replace('(', '').replace(')', '').replace('=', '')

config_backup = open('config.yaml').read()
try:
    for name, cfg in TOPOLOGY_CONFIGS.items():
        with open('config.yaml', 'w') as f:
            yaml.safe_dump(cfg, f)
        out_dir = os.path.join(NETDUMP_DIR, _dir_name(name))
        os.makedirs(out_dir, exist_ok=True)
        print(f'Dumping {name} ...')
        subprocess.run(['java', '-cp', CLASSPATH, 'dynamics.NetworkDump', str(N_SEEDS), out_dir],
                        check=True, capture_output=True, text=True)
finally:
    with open('config.yaml', 'w') as f:
        f.write(config_backup)

restored = subprocess.run(['git', 'diff', '--quiet', 'config.yaml']).returncode == 0
print('Done. config.yaml restored to tracked state:', restored)
assert restored, 'config.yaml was not restored correctly -- check working tree!'

Dumping CNNR ...


Dumping HK (pt=0.0) ...


Dumping HK (pt=0.3) ...


Dumping WS ...


Dumping ER ...


Done. config.yaml restored to tracked state: True


## 2. Compute Table 2's metrics from the dumped edge lists

In [3]:
import networkx as nx
import numpy as np
import pandas as pd
import powerlaw
import warnings
warnings.filterwarnings('ignore')

N_PATH_DIAM_ALPHA_SEEDS = 5  # expensive metrics: restricted to 5 seeds (matches Table 2 caption)

# Only CNNR/HK get a reported power-law exponent (see markdown above / honesty
# caveat below); WS/ER are computed too, for the record, but shown as '---'
# in the final table to match the paper's convention.
HEAVY_TAILED = {'CNNR', 'HK (pt=0.0)', 'HK (pt=0.3)'}


def load_graph(path, n=1000):
    G = nx.DiGraph()
    G.add_nodes_from(range(n))
    with open(path) as f:
        for line in f:
            a, b = line.split()
            G.add_edge(int(a), int(b))
    return G


rows = []
for name in TOPOLOGY_CONFIGS:
    edge_files = sorted(glob.glob(os.path.join(NETDUMP_DIR, _dir_name(name), '*.edges')))
    assert len(edge_files) == N_SEEDS, f'{name}: expected {N_SEEDS} dumps, found {len(edge_files)}'

    avg_degs, clusterings = [], []
    path_lens, diams, alphas = [], [], []
    for i, fpath in enumerate(edge_files):
        G = load_graph(fpath)
        E, N = G.number_of_edges(), G.number_of_nodes()
        avg_degs.append(2 * E / N)  # undirected average degree
        clusterings.append(nx.average_clustering(G))  # directed (Fagiolo 2007)

        if i < N_PATH_DIAM_ALPHA_SEEDS:
            G_und = G.to_undirected()
            giant = max(nx.connected_components(G_und), key=len)
            H = G_und.subgraph(giant)
            path_lens.append(nx.average_shortest_path_length(H))
            diams.append(nx.diameter(H))
            in_deg = [d for _, d in G.in_degree() if d > 0]
            fit = powerlaw.Fit(in_deg, discrete=True, verbose=False)
            alphas.append(fit.power_law.alpha)

    rows.append({
        'topology': name,
        'avg_degree': np.mean(avg_degs),
        'avg_path_length': np.mean(path_lens),
        'diameter': round(np.mean(diams)),
        'power_law_alpha': np.mean(alphas) if name in HEAVY_TAILED else np.nan,
        'avg_clustering': np.mean(clusterings),
    })

tbl_nwmetrics = pd.DataFrame(rows).set_index('topology')
tbl_nwmetrics

,avg_degree,avg_path_length,diameter,power_law_alpha,avg_clustering
topology,,,,,
CNNR,3.9965,5.060982,14,2.264751,0.216187
HK (pt=0.0),4.0000,3.575790,7,2.126644,0.050361
HK (pt=0.3),4.0000,3.258774,7,2.031251,0.213759
WS,4.0000,8.778848,17,NaN,0.184427
ER,3.9913,5.111845,10,NaN,0.001718


## 3. Compare against Table 2 in `docs/journal-June.tex`

Reference values currently in the paper (avg degree, path length, diameter,
power-law exponent, avg clustering): CNNR $4.0, 5.06, 14, 2.26, 0.22$; HK
($p_t{=}0.0$) $4.0, 3.58, 7, 2.13, 0.05$; HK ($p_t{=}0.3$) $4.0, 3.26, 7, 2.03,
0.21$; WS $4.0, 8.78, 17, ---, 0.18$; ER $4.0, 5.11, 10, ---, 0.002$.

**Honesty caveat (carried over from the prior version of this notebook /
project memory):** the likelihood-ratio test of power-law vs. lognormal for
the CNNR/HK in-degree distributions is not significant, i.e. these
distributions are heavy-tailed (beat an exponential null) but are not
statistically distinguishable from a lognormal. The reported $\alpha$ should
therefore be read as "heavy-tailed, power-law-like" rather than "power-law
confirmed." WS/ER fit a much steeper exponent (not shown in the paper's table)
that is not a heavy tail by either standard.